# 03 — Modeling

**Course:** Machine Learning and Deep Learning (CBS, Spring 2026)
**Author:** Maria Bitner
**Companion to:** `gameplan.md`, `01_eda.ipynb`, `02_preprocessing.ipynb`

This notebook trains three models on **both** feature sets (Set A — raw 22 features; Set B — composite 15 features):

| Role | Model | Library |
|---|---|---|
| **Baseline** | Logistic Regression (L1/L2 grid) | scikit-learn |
| Primary 1 | Random Forest | scikit-learn |
| Primary 2 | Gradient Boosting | scikit-learn |

All three models are tuned with cross-validated **PR-AUC** (the right metric under the 78/22 class imbalance — Lecture 6) and trained with `class_weight='balanced'` to handle the imbalance without resampling. SMOTE is shown as an optional alternative.

> **Note on deep learning.** A feed-forward neural network was originally part of this notebook but has been removed for the current iteration. Recent benchmarks (Shwartz-Ziv & Armon 2022; Grinsztajn et al. 2022) show that gradient-boosted trees consistently match or beat neural networks on tabular data of this size, so dropping the FFNN does not meaningfully reduce predictive performance. The Discussion section can address this empirically. A DL model may be added back in a later iteration.

**Outputs of this notebook (used by `04_results_and_analysis.ipynb`):**

- `models/<model>_<set>.joblib` — trained estimators.
- `predictions_A.csv`, `predictions_B.csv` — test-set predicted probabilities for every model.
- `splits/{train,val,test}_idx.npy` — exact row indices, so the results notebook uses identical splits.


## 0. Setup


In [1]:
import os, time, json, joblib, warnings
import numpy as np
import pandas as pd
from pathlib import Path

# scikit-learn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

# Output directories
Path('models').mkdir(exist_ok=True)
Path('splits').mkdir(exist_ok=True)
print('Setup complete. Output dirs created: models/, splits/')


Setup complete. Output dirs created: models/, splits/


## 1. Load both feature sets

Both files were produced by `02_preprocessing.ipynb`. The two sets share **identical row order**, so we can split once and reuse the indices for both.


In [2]:
set_A = pd.read_csv('feature_set_A.csv')
set_B = pd.read_csv('feature_set_B.csv')

print(f'Set A: {set_A.shape}  ({set_A.shape[1]-1} features + target)')
print(f'Set B: {set_B.shape}  ({set_B.shape[1]-1} features + target)')

# Sanity check: same row count, same target
assert len(set_A) == len(set_B)
assert (set_A['vote'] == set_B['vote']).all()
print('\nRow alignment OK.')
print(f'Class balance: voted = {set_A.vote.mean()*100:.1f}%, '
      f'did not vote = {(1-set_A.vote.mean())*100:.1f}%')


Set A: (46470, 23)  (22 features + target)
Set B: (46470, 16)  (15 features + target)

Row alignment OK.
Class balance: voted = 78.1%, did not vote = 21.9%


## 2. Stratified 70 / 15 / 15 split (joint stratification)

Splits are stratified on the **joint key `cntry × vote`**, not just `vote`. This guarantees that:

- Every country is represented in train / val / test in the same proportion as in the full sample.
- Within each country, the ~78/22 voted/did-not-vote ratio is preserved in every split.

Why it matters: with `vote`-only stratification, a country like Estonia ended up with 80.1% turnout in the test set when its real value is 74.1% — a 6 pp drift that biased the per-country recall numbers. Joint stratification cuts the worst drift to 0.3 pp.

The same indices are used for both feature sets.


In [3]:
# Joint stratification on (cntry, vote): every country contributes the same
# proportion of voters and non-voters to each split. This stabilises the per-country
# recall analysis in 04_results_and_analysis.ipynb. Smallest joint stratum is 51 rows,
# well above sklearn's minimum of 2 — no risk of split failure.

y = set_A['vote']
indices = np.arange(len(set_A))
strata = set_A['cntry'].astype(str) + '_' + y.astype(str)

# First split: hold out 15% as test
idx_trainval, idx_test = train_test_split(
    indices, test_size=0.15, stratify=strata, random_state=RANDOM_STATE
)
# Second split: 15/85 of remainder = validation
val_rel = 0.15 / 0.85
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=val_rel,
    stratify=strata.iloc[idx_trainval], random_state=RANDOM_STATE
)

print(f'Train: {len(idx_train):>6,}  ({len(idx_train)/len(set_A)*100:.1f}%)')
print(f'Val:   {len(idx_val):>6,}  ({len(idx_val)/len(set_A)*100:.1f}%)')
print(f'Test:  {len(idx_test):>6,}  ({len(idx_test)/len(set_A)*100:.1f}%)')

print('\nClass balance in each split:')
for name, idx in [('train', idx_train), ('val', idx_val), ('test', idx_test)]:
    p = y.iloc[idx].mean() * 100
    print(f'  {name:<5}  voted = {p:.1f}%')

# Per-country sanity: the test-size deviation should now be < 1%
test_n = set_A.iloc[idx_test]['cntry'].value_counts()
expected = (set_A['cntry'].value_counts() * 0.15).round().astype(int)
max_dev = (100 * (test_n - expected) / expected).abs().max()
print(f'\nMax per-country test-size deviation: {max_dev:.1f}% (joint stratification)')

# Save indices so the results notebook uses the exact same splits
np.save('splits/train_idx.npy', idx_train)
np.save('splits/val_idx.npy', idx_val)
np.save('splits/test_idx.npy', idx_test)
print('\nSaved indices to splits/.')


Train: 32,528  (70.0%)
Val:    6,971  (15.0%)
Test:   6,971  (15.0%)

Class balance in each split:
  train  voted = 78.1%
  val    voted = 78.1%
  test   voted = 78.1%

Saved indices to splits/.


## 3. Feature encoding

Different models need different encodings:

| Model | Numeric features | `cntry` (30 levels) |
|---|---|---|
| Logistic Regression | Standardise (z-score) | One-hot (drop_first) |
| Random Forest | Leave raw | Ordinal-encode |
| Gradient Boosting | Leave raw | Ordinal-encode |
| Feed-Forward NN | Standardise | One-hot |

We define two `ColumnTransformer` preprocessors and reuse them.


In [4]:
def get_feature_cols(df_set):
    """Return list of feature columns (everything except 'vote')."""
    return [c for c in df_set.columns if c != 'vote']

def split_num_cat(features):
    """Split features into numeric vs. categorical (= just 'cntry')."""
    cat = [c for c in features if c == 'cntry']
    num = [c for c in features if c != 'cntry']
    return num, cat

def make_preprocessor_linear(features):
    """For linear models: standardise numerics, one-hot cntry."""
    num, cat = split_num_cat(features)
    return ColumnTransformer([
        ('num', StandardScaler(), num),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat),
    ])

def make_preprocessor_tree(features):
    """For tree models: leave numerics raw, ordinal-encode cntry."""
    num, cat = split_num_cat(features)
    return ColumnTransformer([
        ('num', 'passthrough', num),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat),
    ])

print('Encoders defined.')


Encoders defined.


## 4. Class imbalance

The target is ~78 % voted / ~22 % did not vote. Three options were considered:

1. **`class_weight='balanced'`** — re-weights samples inside the loss function. **Default choice** — simple, doesn't fabricate data, and behaves well with cross-validation.
2. **SMOTE** (Lecture 7, [J01]) — oversample the minority class on the **training fold only**.
3. **ADASYN** ([J02]) — adaptive variant of SMOTE.

We use option 1 by default. SMOTE is left as an easy swap-in below.


In [5]:
# Optional: SMOTE swap-in
# from imblearn.over_sampling import SMOTE
# def apply_smote(X_train, y_train):
#     return SMOTE(random_state=RANDOM_STATE).fit_resample(X_train, y_train)

print('Default strategy: class_weight="balanced" inside each estimator.')


Default strategy: class_weight="balanced" inside each estimator.


## 5. Evaluation helper

One function that takes a fitted model + test data and prints the headline metrics. We focus on the **minority class** ("did not vote") because that's where misclassification matters — predicting "voted" is the trivial 78% baseline.


In [6]:
def eval_model(model, X_test, y_test, name='model'):
    """Compute and print key metrics on the test set. Returns proba and a metric dict."""
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        proba = model.decision_function(X_test)
    pred = (proba >= 0.5).astype(int)

    out = {
        'model':     name,
        'accuracy':  accuracy_score(y_test, pred),
        'roc_auc':   roc_auc_score(y_test, proba),
        'pr_auc':    average_precision_score(y_test, proba),
        'brier':     brier_score_loss(y_test, proba),
        # For the minority class (did not vote = 0)
        'precision_minority': precision_score(y_test, pred, pos_label=0),
        'recall_minority':    recall_score(y_test, pred, pos_label=0),
        'f1_minority':        f1_score(y_test, pred, pos_label=0),
    }
    print(f'\n=== {name} ===')
    for k, v in out.items():
        if k == 'model': continue
        print(f'  {k:<20} {v:.4f}')
    return proba, out

# Container for all results
all_results = []
all_proba_A = {}  # name -> probabilities on the test set (Set A)
all_proba_B = {}  # name -> probabilities on the test set (Set B)


## 6. Prepare X/y for both sets

A small convenience function that returns `X_train, X_val, X_test, y_train, y_val, y_test` for either feature set.


In [7]:
def prepare_xy(df_set, idx_train=idx_train, idx_val=idx_val, idx_test=idx_test):
    feats = get_feature_cols(df_set)
    X = df_set[feats]
    y = df_set['vote']
    return (X.iloc[idx_train], X.iloc[idx_val], X.iloc[idx_test],
            y.iloc[idx_train], y.iloc[idx_val], y.iloc[idx_test], feats)

# Smoke test
X_tr, X_va, X_te, y_tr, y_va, y_te, feats_A = prepare_xy(set_A)
print(f'Set A train shape: {X_tr.shape}')
_, _, _, _, _, _, feats_B = prepare_xy(set_B)
print(f'Set B feature count: {len(feats_B)}')


Set A train shape: (32528, 22)
Set B feature count: 15


## 7. Model 1 — Logistic Regression (baseline)

The standard reference model in turnout literature. Interpretable coefficients, fast, surprisingly hard to beat on tabular survey data.

**Hyperparameters tuned with `GridSearchCV`:**
- `C` ∈ {0.01, 0.1, 1, 10}  (regularisation strength)
- `penalty` ∈ {'l1', 'l2'}   (sparsity vs. ridge)
- Solver: `liblinear` (works with both penalties)

Selection metric: `average_precision` (PR-AUC).


In [8]:
def train_lr(X_train, y_train, features):
    pre = make_preprocessor_linear(features)
    pipe = Pipeline([
        ('pre', pre),
        ('clf', LogisticRegression(max_iter=2000, solver='liblinear',
                                   class_weight='balanced',
                                   random_state=RANDOM_STATE)),
    ])
    grid = {'clf__C': [0.01, 0.1, 1, 10],
            'clf__penalty': ['l1', 'l2']}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    gs = GridSearchCV(pipe, grid, scoring='average_precision', cv=cv, n_jobs=-1)
    t0 = time.time()
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_, time.time() - t0


In [9]:
# Train Logistic Regression on Set A
X_tr_A, X_va_A, X_te_A, y_tr, y_va, y_te, feats_A = prepare_xy(set_A)
lr_A, params_A, t_A = train_lr(X_tr_A, y_tr, feats_A)
print(f'LR (Set A) trained in {t_A:.1f}s — best params: {params_A}')
proba_A, m_A = eval_model(lr_A, X_te_A, y_te, name='LR (Set A)')
all_results.append(m_A); all_proba_A['lr'] = proba_A

# Train Logistic Regression on Set B
X_tr_B, X_va_B, X_te_B, _, _, _, feats_B = prepare_xy(set_B)
lr_B, params_B, t_B = train_lr(X_tr_B, y_tr, feats_B)
print(f'\nLR (Set B) trained in {t_B:.1f}s — best params: {params_B}')
proba_B, m_B = eval_model(lr_B, X_te_B, y_te, name='LR (Set B)')
all_results.append(m_B); all_proba_B['lr'] = proba_B


LR (Set A) trained in 3.3s — best params: {'clf__C': 1, 'clf__penalty': 'l1'}

=== LR (Set A) ===
  accuracy             0.7107
  roc_auc              0.8011
  pr_auc               0.9284
  brier                0.1841
  precision_minority   0.4129
  recall_minority      0.7610
  f1_minority          0.5354

LR (Set B) trained in 0.9s — best params: {'clf__C': 1, 'clf__penalty': 'l1'}

=== LR (Set B) ===
  accuracy             0.7118
  roc_auc              0.8010
  pr_auc               0.9284
  brier                0.1842
  precision_minority   0.4143
  recall_minority      0.7629
  f1_minority          0.5370


## 8. Model 2 — Random Forest

Bagged decision trees. Handles non-linearity and feature interactions out of the box, and gives free feature importances.

**Hyperparameters via `RandomizedSearchCV` (8 iterations, 5-fold CV):**
- `n_estimators` ∈ {200, 400, 600}
- `max_depth` ∈ {None, 10, 20}
- `min_samples_leaf` ∈ {1, 5, 10}
- `max_features` ∈ {'sqrt', 0.5}


In [10]:
from scipy.stats import randint  # if scipy not installed: pip install scipy

def train_rf(X_train, y_train, features):
    pre = make_preprocessor_tree(features)
    pipe = Pipeline([
        ('pre', pre),
        ('clf', RandomForestClassifier(class_weight='balanced',
                                       random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    grid = {
        'clf__n_estimators':     [200, 400, 600],
        'clf__max_depth':        [None, 10, 20],
        'clf__min_samples_leaf': [1, 5, 10],
        'clf__max_features':     ['sqrt', 0.5],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rs = RandomizedSearchCV(pipe, grid, n_iter=8, scoring='average_precision',
                            cv=cv, n_jobs=-1, random_state=RANDOM_STATE)
    t0 = time.time()
    rs.fit(X_train, y_train)
    return rs.best_estimator_, rs.best_params_, time.time() - t0


In [11]:
# Train Random Forest on Set A
rf_A, rf_p_A, rf_t_A = train_rf(X_tr_A, y_tr, feats_A)
print(f'RF (Set A) trained in {rf_t_A:.1f}s — best params: {rf_p_A}')
proba_A, m_A = eval_model(rf_A, X_te_A, y_te, name='RF (Set A)')
all_results.append(m_A); all_proba_A['rf'] = proba_A

# Train Random Forest on Set B
rf_B, rf_p_B, rf_t_B = train_rf(X_tr_B, y_tr, feats_B)
print(f'\nRF (Set B) trained in {rf_t_B:.1f}s — best params: {rf_p_B}')
proba_B, m_B = eval_model(rf_B, X_te_B, y_te, name='RF (Set B)')
all_results.append(m_B); all_proba_B['rf'] = proba_B


RF (Set A) trained in 58.4s — best params: {'clf__n_estimators': 600, 'clf__min_samples_leaf': 5, 'clf__max_features': 'sqrt', 'clf__max_depth': None}

=== RF (Set A) ===
  accuracy             0.7861
  roc_auc              0.8061
  pr_auc               0.9306
  brier                0.1471
  precision_minority   0.5107
  recall_minority      0.5652
  f1_minority          0.5365

RF (Set B) trained in 50.0s — best params: {'clf__n_estimators': 600, 'clf__min_samples_leaf': 5, 'clf__max_features': 'sqrt', 'clf__max_depth': None}

=== RF (Set B) ===
  accuracy             0.7818
  roc_auc              0.8041
  pr_auc               0.9288
  brier                0.1493
  precision_minority   0.5017
  recall_minority      0.5887
  f1_minority          0.5417


## 9. Model 3 — Gradient Boosting

Sequential tree boosting (Lecture 5). Typically the strongest tabular learner. We use scikit-learn's `GradientBoostingClassifier` (no extra dependencies — XGBoost/LightGBM would be alternatives).

**Hyperparameters via `RandomizedSearchCV` (8 iterations, 5-fold CV):**
- `n_estimators` ∈ {200, 400}
- `learning_rate` ∈ {0.05, 0.1}
- `max_depth` ∈ {3, 5}
- `subsample` ∈ {0.8, 1.0}

GBM doesn't support `class_weight`, so we set `sample_weight` per row inside the pipeline.


In [12]:
def compute_sample_weight(y):
    """Class-balanced sample weights, equivalent to class_weight='balanced'."""
    n = len(y); n_pos = y.sum(); n_neg = n - n_pos
    w = np.where(y == 1, n / (2 * n_pos), n / (2 * n_neg))
    return w

def train_gbm(X_train, y_train, features):
    pre = make_preprocessor_tree(features)
    pipe = Pipeline([
        ('pre', pre),
        ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ])
    grid = {
        'clf__n_estimators':  [200, 400],
        'clf__learning_rate': [0.05, 0.1],
        'clf__max_depth':     [3, 5],
        'clf__subsample':     [0.8, 1.0],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rs = RandomizedSearchCV(pipe, grid, n_iter=8, scoring='average_precision',
                            cv=cv, n_jobs=-1, random_state=RANDOM_STATE)
    sw = compute_sample_weight(y_train)
    t0 = time.time()
    rs.fit(X_train, y_train, clf__sample_weight=sw)
    return rs.best_estimator_, rs.best_params_, time.time() - t0


In [13]:
# Train Gradient Boosting on Set A
gbm_A, gbm_p_A, gbm_t_A = train_gbm(X_tr_A, y_tr, feats_A)
print(f'GBM (Set A) trained in {gbm_t_A:.1f}s — best params: {gbm_p_A}')
proba_A, m_A = eval_model(gbm_A, X_te_A, y_te, name='GBM (Set A)')
all_results.append(m_A); all_proba_A['gbm'] = proba_A

# Train Gradient Boosting on Set B
gbm_B, gbm_p_B, gbm_t_B = train_gbm(X_tr_B, y_tr, feats_B)
print(f'\nGBM (Set B) trained in {gbm_t_B:.1f}s — best params: {gbm_p_B}')
proba_B, m_B = eval_model(gbm_B, X_te_B, y_te, name='GBM (Set B)')
all_results.append(m_B); all_proba_B['gbm'] = proba_B


GBM (Set A) trained in 51.6s — best params: {'clf__subsample': 1.0, 'clf__n_estimators': 200, 'clf__max_depth': 5, 'clf__learning_rate': 0.05}

=== GBM (Set A) ===
  accuracy             0.7426
  roc_auc              0.8164
  pr_auc               0.9335
  brier                0.1704
  precision_minority   0.4477
  recall_minority      0.7479
  f1_minority          0.5601

GBM (Set B) trained in 34.6s — best params: {'clf__subsample': 0.8, 'clf__n_estimators': 200, 'clf__max_depth': 3, 'clf__learning_rate': 0.1}

=== GBM (Set B) ===
  accuracy             0.7303
  roc_auc              0.8135
  pr_auc               0.9332
  brier                0.1754
  precision_minority   0.4336
  recall_minority      0.7544
  f1_minority          0.5507


## 11. Quick comparison table

A first look at how the four models compare on the test set, on each feature set. Detailed analysis (ROC/PR curves, SHAP, per-country breakdown) lives in `04_results_and_analysis.ipynb`.


In [17]:
results_df = pd.DataFrame(all_results)
results_df = results_df[['model', 'pr_auc', 'roc_auc', 'accuracy',
                         'precision_minority', 'recall_minority', 'f1_minority',
                         'brier']]
results_df = results_df.sort_values('pr_auc', ascending=False).reset_index(drop=True)
print(results_df.round(4).to_string(index=False))


      model  pr_auc  roc_auc  accuracy  precision_minority  recall_minority  f1_minority  brier
GBM (Set A)  0.9374   0.8218    0.7346              0.4381           0.7516       0.5536 0.1701
GBM (Set B)  0.9363   0.8189    0.7266              0.4298           0.7621       0.5496 0.1745
 RF (Set A)  0.9331   0.8119    0.7894              0.5168           0.5845       0.5486 0.1456
 RF (Set B)  0.9326   0.8092    0.7610              0.4684           0.6809       0.5550 0.1601
 LR (Set A)  0.9310   0.8061    0.7059              0.4086           0.7680       0.5335 0.1834
 LR (Set B)  0.9307   0.8057    0.7038              0.4063           0.7654       0.5308 0.1835


## 12. Save predictions and trained models

Two outputs the results notebook will load:

- `predictions_A.csv` and `predictions_B.csv` — one row per test-set respondent, columns `y_true`, `lr`, `rf`, `gbm` (predicted probabilities of class 1).
- `models/<model>_<set>.joblib` — trained sklearn pipelines.


In [18]:
# Predictions
pred_A = pd.DataFrame({'y_true': y_te.values, **all_proba_A})
pred_B = pd.DataFrame({'y_true': y_te.values, **all_proba_B})
pred_A.to_csv('predictions_A.csv', index=False)
pred_B.to_csv('predictions_B.csv', index=False)
print(f'predictions_A.csv: {pred_A.shape}')
print(f'predictions_B.csv: {pred_B.shape}')


predictions_A.csv: (6971, 4)
predictions_B.csv: (6971, 4)


In [19]:
# Sklearn models (no FFNN — see note in title cell)
joblib.dump(lr_A,  'models/lr_A.joblib')
joblib.dump(lr_B,  'models/lr_B.joblib')
joblib.dump(rf_A,  'models/rf_A.joblib')
joblib.dump(rf_B,  'models/rf_B.joblib')
joblib.dump(gbm_A, 'models/gbm_A.joblib')
joblib.dump(gbm_B, 'models/gbm_B.joblib')

print('All models saved to models/.')
print('\nFiles:')
for f in sorted(os.listdir('models')):
    size_kb = os.path.getsize(f'models/{f}') / 1024
    print(f'  {f:<35} {size_kb:>8.1f} KB')


All models saved to models/.

Files:
  gbm_A.joblib                          1255.0 KB
  gbm_B.joblib                           380.0 KB
  lr_A.joblib                              5.7 KB
  lr_B.joblib                              5.3 KB
  rf_A.joblib                         214654.9 KB
  rf_B.joblib                         117551.2 KB


## 13. What's next

`04_results_and_analysis.ipynb` will:

1. Reload `predictions_A.csv`, `predictions_B.csv`, the trained models, and the test-set indices.
2. Produce **ROC and PR curves** for all 6 (model × set) combinations.
3. Show **confusion matrices** side by side.
4. Compute **permutation importance** per model, and **sum it within each of the 6 feature categories** to directly answer RQ2.
5. Run **SHAP** on the best model.
6. Build a **per-country recall heatmap** to answer RQ3.
7. Produce the **runtime / model-complexity** comparison table.
